# 02 — Data I/O and Cleaning

Reading the common file formats correctly, handling missing data, deduplication, and the memory-optimization tricks (`category` dtype, downcasting) that come up whenever an interview asks "the CSV is 10 GB and won't fit in RAM — what do you do?"

In [1]:
import pandas as pd
import numpy as np
import io

## 1. Reading CSV/JSON/Parquet correctly

- `pd.read_csv(..., dtype={...}, parse_dates=[...])` — always pass explicit `dtype` for columns pandas might misinfer (e.g. a zip-code or ID column that looks numeric but should stay a zero-padded string), and `parse_dates` instead of parsing dates after the fact.
- `pd.read_parquet(...)` — Parquet carries its own schema and is columnar, so reads are faster and don't need `dtype`/`parse_dates` hints; prefer it over CSV for anything beyond raw ingestion (same reasoning as the Spark notebooks' format comparison).
- `pd.read_json(..., lines=True)` for newline-delimited JSON (the common log/event format), not the default (a single JSON array/object).

In [2]:
csv_text = """user_id,zip_code,amount,signup_date
1,07030,19.99,2024-01-05
2,00501,29.99,2024-02-10
3,90210,9.99,2024-03-15
"""

# Without dtype hints, zip_code ("07030") loses its leading zero as int64
naive = pd.read_csv(io.StringIO(csv_text))
print(naive.dtypes)
print(naive["zip_code"].tolist())     # [7030, 501, 90210] -- leading zeros GONE

correct = pd.read_csv(io.StringIO(csv_text), dtype={"zip_code": "string"}, parse_dates=["signup_date"])
print(correct.dtypes)
print(correct["zip_code"].tolist())   # ['07030', '00501', '90210'] -- preserved

user_id          int64
zip_code         int64
amount         float64
signup_date        str
dtype: object
[7030, 501, 90210]
user_id                 int64
zip_code               string
amount                float64
signup_date    datetime64[us]
dtype: object
['07030', '00501', '90210']


## 2. Missing data

- `df.isna().sum()` — count nulls per column, always the first check on a new dataset.
- `df.dropna(subset=[...])` — drop rows missing required fields.
- `df.fillna(value)` — fill with a constant, or a per-column dict; `df["col"].fillna(df["col"].mean())` for numeric imputation.
- `df.interpolate()` — fill gaps by interpolating between known values, useful for ordered/time-series data specifically (don't use on unordered categorical data).

In [3]:
messy = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "age": [25, np.nan, 40, np.nan],
    "country": ["US", "US", None, "IN"],
})

print(messy.isna().sum())

filled = messy.copy()
filled["age"] = filled["age"].fillna(filled["age"].mean())
filled["country"] = filled["country"].fillna("unknown")
filled

user_id    0
age        2
country    1
dtype: int64


,user_id,age,country
0,1,25.0,US
1,2,32.5,US
2,3,40.0,unknown
3,4,32.5,IN


**Interview trap:** `fillna(0)` on a numeric column silently treats "missing" and "actually zero" as the same thing — fine for a count column where absence genuinely means zero, wrong for something like `age` or `price` where 0 is a real, different value. Always ask what the missingness *means* before picking a fill strategy.

## 3. Duplicates

`df.duplicated(subset=[...], keep="first"/"last"/False)` flags duplicate rows; `df.drop_duplicates(...)` removes them. `keep="last"` combined with a sort by timestamp is the standard pandas pattern for "keep only the latest record per key" (the same problem solved with a window function in the Spark notebooks).

In [4]:
updates = pd.DataFrame({
    "user_id": [1, 1, 2, 2],
    "email": ["a@x.com", "a@new.com", "b@x.com", "b@older.com"],
    "updated_at": pd.to_datetime(["2024-01-01", "2024-03-05", "2024-02-10", "2024-01-20"]),
})

latest_per_user = (
    updates.sort_values("updated_at")
    .drop_duplicates(subset="user_id", keep="last")
    .sort_values("user_id")
    .reset_index(drop=True)
)
latest_per_user

,user_id,email,updated_at
0,1,a@new.com,2024-03-05
1,2,b@x.com,2024-02-10


## 4. Memory optimization — the "10 GB CSV" interview question

1. **`category` dtype** for low-cardinality string columns (a `plan` column with 3 distinct values stored as `object` repeats the full string per row; `category` stores it as an integer code + a lookup table once).
2. **Downcast numeric types** — `pd.to_numeric(col, downcast="integer")` or `"float"` shrinks `int64`/`float64` to the smallest dtype that still fits the data's actual range.
3. **`chunksize` in `read_csv`** — process a file larger than RAM in chunks, aggregating incrementally instead of loading it all at once.
4. **Beyond that: it's not a pandas job anymore** — pandas is single-machine, in-memory. If the data doesn't fit even after these tricks, that's the interview signal to say "I'd reach for chunked/streaming processing, Polars (lazy, out-of-core), or Spark" rather than trying to force pandas to scale.

In [5]:
plans = pd.DataFrame({"plan": np.random.choice(["free", "pro", "enterprise"], size=100_000)})
print("object dtype memory:", plans.memory_usage(deep=True).sum(), "bytes")

plans["plan"] = plans["plan"].astype("category")
print("category dtype memory:", plans.memory_usage(deep=True).sum(), "bytes")

object dtype memory: 6266535 bytes
category dtype memory: 100320 bytes


In [6]:
# Chunked processing pattern for a file too large to load at once
total_amount = 0.0
for chunk in pd.read_csv(io.StringIO(csv_text), chunksize=2):
    total_amount += chunk["amount"].sum()
print("total across chunks:", total_amount)

total across chunks: 59.97


## 5. Interview Q&A

1. **"A 10 GB CSV needs to be processed on a machine with 8 GB RAM — how?"** — `chunksize` to stream and aggregate incrementally, `category`/downcasting to shrink what you do hold in memory, or escalate to Polars/Spark/Dask if it's a recurring, not one-off, need.
2. **"Why explicitly pass `dtype` to `read_csv` instead of letting it infer?"** — inference can silently corrupt data (leading zeros in IDs/zip codes lost to int coercion) and costs an extra scan; also makes the read faster and the schema self-documenting.
3. **"`fillna(0)` vs `fillna(mean)` vs `dropna()` — how do you choose?"** — depends on what missingness *means* for that column: true zero (fillna(0)), a plausible estimate for a numeric feature going into an aggregate/model (fillna(mean/median)), or a hard requirement you can't proceed without (dropna).
4. **"How do you keep only the latest row per key?"** — sort by the timestamp column, then `drop_duplicates(subset=key, keep="last")`.

## Summary

- Pass explicit `dtype`/`parse_dates` to `read_csv`; prefer Parquet for anything past raw ingestion.
- Decide what missing data *means* before choosing `fillna`/`dropna`/`interpolate`.
- `sort_values` + `drop_duplicates(keep="last")` = keep-latest-per-key.
- `category` dtype + downcasting shrink memory; `chunksize` handles files bigger than RAM; beyond that, it's not pandas's job anymore.
- Next: `03_indexing_selection_and_transformation.ipynb`.